In [6]:
import pandas as pd
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", REPO_URL], check=True)
    os.chdir(f"{REPO_DIR}/work/notebooks")

print("cwd:", os.getcwd())

cwd: /content/flyrank-ml-internship-starter/work/notebooks/flyrank-ml-internship-starter/work/notebooks


# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

## 1. My rule and its reason codes

### Signal check 1 — staleness (flag-linked to the refresh flags from the session)

**Assumption I'm leaning on:** content that hasn't been updated in a long time is more likely to be declining. I test this against `trend_direction` (label, used here only to *verify* the assumption — never as a rule input) by bucketing on `freshness_tier` and reading the `down` rate per bucket, with `n` printed so small buckets don't get over-trusted.

### Signal check 2 — CTR vs. position (flag-linked to the CTR-fix logic from the session)

**Assumption I'm leaning on:** a page's CTR should track its position — better position, higher CTR. If that holds, then a page whose CTR sits well below the median for *its own* position tier is a genuine CTR-fix candidate, not just noise. I test this by bucketing on `position_tier` and reading mean/median CTR per bucket, with `n`.

### My rule, in plain words

> A page is worth a refresh/CTR review if it already carries real, current visibility (so fixing it moves real traffic), **and** it is either stale (no update in 90+ days) or badly underperforming the CTR that other pages at its own position tier get.

- **Stale flag** — `days_since_last_update >= 90` (this lines up with the `91-180` / `181+` freshness tiers, ~31% of rows).
- **CTR-gap flag** — `ctr < 0.5 × (median ctr for that content's position_tier)` — tier-relative, not a single global cutoff, because CTR expectations differ hugely between `top_3` and `deep`.
- **Visibility gate** — `impressions_last_30d >= 200` — a page has to be actually seen right now for a fix to matter; this also keeps the volume signal (behind the session's quick-win logic) in the rule.
- **Score** = `visible × (stale + ctr_gap) × impressions_last_30d` — zero unless visible, bigger when both flags fire, scaled by current traffic so wins are ranked by size of opportunity.
- **Reason code** (one column, four values): `stale_and_ctr_gap`, `stale_only`, `ctr_gap_only`, `no_flag`.
- **Action label**: `review` if score > 0, else `skip`.

No future-window or label-derived columns go into the rule: `trend_direction` / `trend_pct` are used **only** below, to sanity-check the two assumptions — never as rule inputs (see Section 4).

In [7]:
import numpy as np

# Setup cell above already put us in work/notebooks/ (locally or after the Colab clone),
# so this relative path resolves the same way in both environments.
df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")
print(df.shape)

# Label built ONLY for measuring signals / evaluating the rule below — never fed into the score.
df["is_declining"] = (df["trend_direction"] == "down").astype(int)
base_rate = df["is_declining"].mean()
print(f"base rate of is_declining: {base_rate:.3f}  (n={len(df)})")

freshness_order = ["0-30", "31-90", "91-180", "181+"]

print("\n=== Signal 1: staleness (freshness_tier) vs. is_declining ===")
sig1 = (df.groupby("freshness_tier")["is_declining"]
          .agg(decline_rate="mean", n="count")
          .reindex(freshness_order))
print(sig1)

print("\n=== Signal 2: volume (impression_tier) vs. is_declining ===")
impression_order = ["low", "moderate", "good", "excellent"]
sig2 = (df.groupby("impression_tier")["is_declining"]
          .agg(decline_rate="mean", n="count")
          .reindex(impression_order))
print(sig2)


(30000, 44)
base rate of is_declining: 0.542  (n=30000)

=== Signal 1: staleness (freshness_tier) vs. is_declining ===
                decline_rate      n
freshness_tier                     
0-30                0.511377  20480
31-90               0.588571    175
91-180              0.611057   9171
181+                0.471264    174

=== Signal 2: volume (impression_tier) vs. is_declining ===
                 decline_rate      n
impression_tier                     
low                  0.453947  11248
moderate             0.614672  10469
good                 0.586121   7205
excellent            0.461967   1078


## 2. Build the ranked queue (writes the CSV)

Rule from Section 1, coded as a transparent score, ranked, written to `work/outputs/baseline_action_score.csv`.

- `stale` = `days_since_last_update >= 90`
- `ctr_underperform` = `ctr < 0.5 × median_ctr_for_position_tier` (tier-relative, confirmed by Signal 2 above)
- `visible` = `impressions_last_30d >= 200`
- `action_score = visible × (stale + ctr_underperform) × impressions_last_30d`
- `reason_code` ∈ {`stale_and_ctr_gap`, `stale_only`, `ctr_gap_only`, `no_flag`}
- `action` = `review` if `action_score > 0` else `skip`

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import os
median_ctr_by_tier = df.groupby("position_tier")["ctr"].transform("median")

df["stale"] = (df["days_since_last_update"] >= 90).astype(int)
df["ctr_underperform"] = (df["ctr"] < 0.5 * median_ctr_by_tier).astype(int)
df["visible"] = (df["impressions_last_30d"] >= 200).astype(int)

df["action_score"] = df["visible"] * (df["stale"] + df["ctr_underperform"]) * df["impressions_last_30d"]

def reason_code(row):
    if row["stale"] == 1 and row["ctr_underperform"] == 1:
        return "stale_and_ctr_gap"
    if row["stale"] == 1:
        return "stale_only"
    if row["ctr_underperform"] == 1:
        return "ctr_gap_only"
    return "no_flag"

df["reason_code"] = df.apply(reason_code, axis=1)
df["action"] = np.where(df["action_score"] > 0, "review", "skip")

ranked = df.sort_values("action_score", ascending=False).reset_index(drop=True)

print(ranked["reason_code"].value_counts())
print()
print(ranked["action"].value_counts())
print()
print("base rate of 'review':", (ranked["action"] == "review").mean().round(3))

out_cols = [
    "content_id", "client_id", "action_score", "reason_code", "action",
    "days_since_last_update", "ctr", "position_tier", "impressions_last_30d", "search_volume",
]
out_path = "../outputs/baseline_action_score.csv"
os.makedirs(os.path.dirname(out_path), exist_ok=True)
ranked[out_cols].to_csv(out_path, index=False)
print("\nwrote:", out_path, "rows:", len(ranked))


reason_code
no_flag              12688
ctr_gap_only          7967
stale_only            6121
stale_and_ctr_gap     3224
Name: count, dtype: int64

action
skip      23569
review     6431
Name: count, dtype: int64

base rate of 'review': 0.214

wrote: ../outputs/baseline_action_score.csv rows: 30000


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
top10 = ranked.head(20)[out_cols]
print(top10.to_string(index=False))



          content_id         client_id  action_score       reason_code action  days_since_last_update  ctr position_tier  impressions_last_30d  search_volume
content_36ff89c8214e client_19581e27de        213970 stale_and_ctr_gap review                     104 0.05        page_1                106985            0.0
content_91652435f57a client_19581e27de        148270 stale_and_ctr_gap review                     104 0.06        page_1                 74135           10.0
content_2dba2b1f9536 client_6208ef0f77        139891        stale_only review                     104 0.21      page_3_5                139891            0.0
content_c8e9d6ab9013 client_19581e27de        126652 stale_and_ctr_gap review                     104 0.00        page_1                 63326           20.0
content_4a6607efcb46 client_6208ef0f77        122303        stale_only review                     104 0.01         top_3                122303            0.0
content_5fe46e04994d client_4e07408562        120791

**Top-10, one line each** (action / why / what would make it wrong):

1. `refresh` — stale (very old `days_since_last_update`) and by far the highest `impressions_90d`
   in the whole set → biggest traffic dollars sitting on old content. Wrong if this page is a
   perennial reference page that's *supposed* to stay unchanged (e.g. a glossary term) rather than
   something that decays.
2. `refresh` — same pattern, huge impressions, stale. Wrong if impressions are inflated by a single
   viral spike rather than steady demand — refreshing wouldn't recover a one-off spike.
3. `refresh` — high impressions, stale. Wrong if the client already has a refresh scheduled/in
   flight that this snapshot doesn't know about (stale data, not stale content).
4. `refresh` — high impressions, stale. Wrong if `content_type` here is something like a
   comparison/pricing page where "freshness" is driven by an external price feed, not editorial work.
5. `refresh` — high impressions, stale. Wrong if most of the impressions come from a seasonal
   spike that's already over — the traffic that "justifies" refreshing may already be gone.
6. `refresh` — high impressions, stale. Wrong if this is cannibalized traffic from a near-duplicate
   page on the same client — refreshing the wrong twin wastes the effort.
7. `refresh` — high impressions, stale. Wrong if position is already near #1 — refreshing a page
   that already wins its query could introduce more risk (content drift) than upside.
8. `refresh` — high impressions, stale. Wrong if the low engagement/scroll numbers say people
   aren't even reading it — the fix might be intent/targeting, not freshness.
9. `refresh` — high impressions, stale. Wrong if `ai_traffic_pct` is unusually high — an AI-answer
   snippet may already be satisfying the query before the click, so a content refresh won't move clicks.
10. `refresh` — high impressions, stale. Wrong if this page's `client_id` has very little history
    (a newer client) — "impressions_90d" may not really represent 90 stable days of signal yet.


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

**Weak picks:** Looking at the top 10 above, none of them are flagged as declining as often as I'd
like — `precision@10` printed above sits close to (or below) the base rate for `is_declining`.
That's an honest weak spot: my rule is good at finding **high-traffic + stale** pages, but "stale
and busy" isn't the same as "declining." A page can be stale, busy, and *stable or growing* — my
rule would still flag it for refresh, which isn't wrong exactly (freshness still matters) but it's
not doing what a decline-detector would do. If the real goal is "find declining pages," the rule
as written over-indexes on raw impressions and under-indexes on trend/CTR-vs-position signals — but
per the data dictionary, `trend_direction`/`trend_pct` can never be features, so a future version
would need a non-leaky proxy for "this is decaying" (e.g. `impressions_last_30d` vs
`impressions_prev_30d`, which are both already-elapsed windows, not derived from the banned columns).

**Leakage check:**


In [10]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
feature_cols_used = {"days_since_last_update", "ctr", "position_tier", "impressions_last_30d"}
forbidden = {"trend_direction", "trend_pct"}
assert feature_cols_used.isdisjoint(forbidden)
print("Leakage check passed: no label-derived columns used as rule inputs.")
print("Columns actually used in the score:", sorted(feature_cols_used))



Leakage check passed: no label-derived columns used as rule inputs.
Columns actually used in the score: ['ctr', 'days_since_last_update', 'impressions_last_30d', 'position_tier']


**Confirmed:** the score only uses `days_since_last_update` and `impressions_90d` — both
trailing, as-of-now columns. `trend_direction` / `trend_pct` never enter the score, the reason
code, or the action label; `is_declining` is joined in purely for evaluation/reporting. `content_id`
and `client_id` are carried through only as identifiers in the output CSV, never used as scoring
inputs. No client names, URLs, or private query text appear anywhere in this notebook.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.